# SeaDronesSee ConvNeXt PT2E QAT — Kaggle

Notebook này chạy luồng PT2E graph-mode cho ConvNeXt backbone, rồi convert và benchmark INT8 trên CPU.

Điểm chính:

- PT2E chỉ quantize ConvNeXt body mặc định (`quantization.pt2e.region = backbone`)
- train theo từng epoch nhỏ để dễ resume trên Kaggle
- cuối cùng benchmark FP32 vs PT2E INT8


In [ ]:
import os
import sys
import json
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

import torch
import yaml

REPO = Path('/kaggle/working/EchteAI')
REPO_URL = os.environ.get('ECHTEAI_REPO_URL', 'https://github.com/NguyenDucThang-tb/EchteAI.git')
if not REPO.exists():
    print(f'Repo not found at {REPO}; cloning from {REPO_URL} ...', flush=True)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    print(f'Repo already present at {REPO}', flush=True)

sys.path.insert(0, str(REPO))

print('Repo:', REPO)
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco,pt2e]'
], check=True, cwd=REPO)
print('Dependencies installed')


In [ ]:
from pipelines.convnext_qat.checkpoint import checkpoint_size_mb


def run_and_log(command, log_path, cwd=REPO):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', ' '.join(command), flush=True)
    print('Persistent log:', log_path, flush=True)
    print('Started:', datetime.now().isoformat(timespec='seconds'), flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'
===== START {datetime.now().isoformat()} =====
')
        log_file.write(' '.join(command) + '
')
        log_file.flush()
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env, cwd=cwd,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
        log_file.write(f'===== END code={return_code} {datetime.now().isoformat()} =====
')
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def checkpoint_epoch(path):
    path = Path(path)
    if not path.is_file():
        return 0
    payload = torch.load(path, map_location='cpu', weights_only=False)
    return int(payload.get('epoch', 0)) if isinstance(payload, dict) else 0


def find_latest_checkpoint(root, filename):
    root = Path(root)
    matches = list(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Không tìm thấy {filename} trong {root}')
    scored = []
    for path in matches:
        try:
            scored.append((checkpoint_epoch(path), path))
        except Exception:
            continue
    if not scored:
        raise FileNotFoundError(f'Không đọc được checkpoint nào cho {filename} trong {root}')
    scored.sort(key=lambda item: (item[0], str(item[1])))
    return scored[-1][1]


def find_dataset_root(root):
    root = Path(root)
    matches = list(root.rglob('instances_train.json'))
    if not matches:
        raise FileNotFoundError(f'Không tìm thấy instances_train.json trong {root}')
    for ann in matches:
        candidate = ann.parent.parent
        if (candidate / 'images' / 'train').exists() and (candidate / 'images' / 'val').exists():
            return candidate
    return matches[0].parent.parent


In [ ]:
DATA_ROOT = find_dataset_root('/kaggle/input')
FP32_SOURCE = find_latest_checkpoint('/kaggle/input', 'fp32_last.pt')
SELECTIVE_INT8_SOURCE = None
try:
    SELECTIVE_INT8_SOURCE = find_latest_checkpoint('/kaggle/input', 'selective_int8.pt')
except FileNotFoundError:
    pass

print('DATA_ROOT:', DATA_ROOT)
print('Using FP32 source:', FP32_SOURCE)
print('FP32 epoch:', checkpoint_epoch(FP32_SOURCE))
print('Optional selective INT8 source:', SELECTIVE_INT8_SOURCE)
assert checkpoint_epoch(FP32_SOURCE) >= 5, 'Cần checkpoint FP32 epoch 5 hoặc lớn hơn'


In [ ]:
WORK = Path('/kaggle/working/seadronessee_pt2e_run')
OUTPUT = WORK / 'checkpoints'
LOGS = WORK / 'logs'
OUTPUT.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

base = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
base['dataset'].update({
    'train_images': str(DATA_ROOT / 'images/train'),
    'train_annotations': str(DATA_ROOT / 'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT / 'images/val'),
    'val_annotations': str(DATA_ROOT / 'annotations/instances_val.json'),
    'test_images': str(DATA_ROOT / 'images/val'),
    'test_annotations': str(DATA_ROOT / 'annotations/instances_val.json'),
})
base['training']['batch_size'] = 2
base['training']['fp32_batch_size'] = 2
base['training']['qat_batch_size'] = 2
base['training']['fp32_epochs'] = 5
base['training']['pt2e_qat_epochs'] = 3
base['training']['epoch_benchmark_images'] = 100
base['quantization']['variant'] = 'M3'
base['quantization']['pt2e']['region'] = 'backbone'
base['quantization']['pt2e']['observer_warmup_epochs'] = 1
base['quantization']['pt2e']['observer_freeze_epochs'] = 1
base['quantization']['pt2e']['example_batch_size'] = 2
base['quantization']['pt2e']['maximum_batch_size'] = 2
base['output'] = {
    'directory': str(OUTPUT),
    'fp32_best': str(FP32_SOURCE),
    'fp32_last': str(FP32_SOURCE),
    'pt2e_qat_best': str(OUTPUT / 'pt2e_qat_best.pt'),
    'pt2e_qat_last': str(OUTPUT / 'pt2e_qat_last.pt'),
    'pt2e_int8_model': str(OUTPUT / 'pt2e_int8.pt'),
    'pt2e_int8_evaluation': str(OUTPUT / 'pt2e_int8_evaluation.json'),
    'pt2e_benchmark_json': str(OUTPUT / 'pt2e_benchmark.json'),
}
RUNTIME_CONFIG = WORK / 'runtime.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False), encoding='utf-8')

print('Runtime config:', RUNTIME_CONFIG)
print(RUNTIME_CONFIG.read_text())
print('PT2E output dir:', OUTPUT)


In [ ]:
def run_pt2e_one_epoch(epoch_idx):
    pt2e_last = OUTPUT / 'pt2e_qat_last.pt'
    command = [
        sys.executable, '-u', 'scripts/train_next_epoch.py',
        '--config', str(RUNTIME_CONFIG),
        '--stage', 'pt2e',
    ]
    if pt2e_last.exists():
        command += ['--resume', str(pt2e_last)]
    print(f'PT2E epoch {epoch_idx} command ready')
    run_and_log(command, LOGS / 'pt2e_train.log', cwd=REPO)

print('PT2E training will run single-GPU by design in this repo (graph-mode PT2E).')


In [ ]:
# Chạy 3 epoch PT2E liên tiếp (mỗi lần 1 epoch để dễ resume trên Kaggle)
for i in range(3):
    print(f'===== PT2E epoch {i + 1}/3 =====')
    run_pt2e_one_epoch(i + 1)
    pt2e_last = OUTPUT / 'pt2e_qat_last.pt'
    print('Checkpoint epoch:', checkpoint_epoch(pt2e_last))
    print('Checkpoint size MB:', f'{checkpoint_size_mb(pt2e_last):.2f}')


In [ ]:
# Benchmark PT2E INT8 sau khi convert xong
pt2e_int8 = OUTPUT / 'pt2e_int8.pt'
if not pt2e_int8.is_file():
    print('Chưa có pt2e_int8.pt; hãy để train_pt2e_qat.py chạy đến phase frozen rồi benchmark lại.')
else:
    command = [
        sys.executable, '-u', 'scripts/benchmark_pt2e.py',
        '--config', str(RUNTIME_CONFIG),
        '--fp32-checkpoint', str(FP32_SOURCE),
        '--pt2e-int8-checkpoint', str(pt2e_int8),
        '--output', str(OUTPUT / 'pt2e_benchmark.json'),
    ]
    if SELECTIVE_INT8_SOURCE is not None:
        command += ['--eager-int8-checkpoint', str(SELECTIVE_INT8_SOURCE)]
    run_and_log(command, LOGS / 'pt2e_benchmark.log', cwd=REPO)
    print((OUTPUT / 'pt2e_benchmark.json').read_text())


## Ghi chú

- Notebook này tập trung vào PT2E graph-mode để tối ưu inference speed.
- Default region là `backbone`; nhánh `backbone_fpn` vẫn có thể bật trong `runtime.yaml` nhưng chưa phải default ổn định nhất.
- PT2E train hiện vẫn là single-GPU theo script của repo; mục tiêu chính của notebook này là benchmark/convert ra PT2E INT8 nhanh hơn.
